In [ ]:
from pathlib import Path
import os
import subprocess

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "src" / "train_model.py").exists():
    repo_root = repo_root.parent
if not (repo_root / "src" / "train_model.py").exists():
    repo_root = Path("/content/09_12523118_10123078_CustomerChurn")
    if not repo_root.exists():
        subprocess.run(["git", "clone", "https://github.com/TaDucHuy0810/09_12523118_10123078_CustomerChurn.git", str(repo_root)], check=True)
os.chdir(repo_root)
print("Project root:", repo_root)

# 02 - Tiền xử lý dữ liệu

Notebook dùng chung hàm load/preprocess với pipeline huấn luyện chính để tránh lệch schema. Split được tạo trước; imputer, scaler và encoder chỉ fit trên tập train.

In [ ]:
from pathlib import Path
import sys
from sklearn.model_selection import train_test_split
from IPython.display import display

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / 'src' / 'train_model.py').exists():
    project_root = project_root.parent
if not (project_root / 'src' / 'train_model.py').exists():
    raise RuntimeError('Clone the repository and set the working directory to its root before running this notebook.')
sys.path.insert(0, str(project_root))
from src.train_model import RANDOM_STATE, load_dataset, make_preprocessor

X, y = load_dataset()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
preprocessor = make_preprocessor(X)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
print('Cleaned rows:', len(X))
print('Features:', len(X.columns))
print('Train/test rows:', len(X_train), len(X_test))
print('Transformed train/test shapes:', X_train_processed.shape, X_test_processed.shape)
display(X.head())

## Quyết định tiền xử lý

- Chuẩn hóa dấu phẩy thập phân ở Monthly Charges và Total Charges trước khi chuyển sang số.
- Loại ID, địa lý và target-derived fields để giảm leakage.
- Dùng stratified 80/20 split do nhãn churn mất cân bằng.
- Median imputation + StandardScaler cho số; most-frequent imputation + OneHotEncoder cho category.
- Fit preprocessing trên train only; test chỉ được transform.